# Cross-Model Layer-wise Probe Ablation

**Is the deployed 3-layer (triple) hallucination probe worth its extra parameters, or
does a smaller set of transformer depths do just as well across models?**

This notebook is the **cross-model synthesis** of the three per-model ablation
notebooks:

- `qwen_colab_a100_ragtruth18k__Layer_wise_probe_ablation.ipynb`
- `mistral_colab_a100_ragtruth18k__Layer_wise_probe_ablation.ipynb`
- `llama_colab_a100_ragtruth18k__Layer_wise_probe_ablation.ipynb`

It is deliberately **lean and CPU-only**: it loads **no model weights, no RAG/FAISS
index, runs no training and no HaluEval scoring**. It only embeds the already-executed
numbers and draws the comparison. It runs in seconds on a laptop.

## Protocol (identical across models)
- **Dataset:** RAGTruth, **4000 rows**, stratified train/val split.
- **Signals per layer:** **CEV** (decoder-block output / residual stream) and **IAV**
  (input to the MLP `down_proj`), each pooled to the **last token**; two probes are
  trained and **fused** (val-tuned weighted average).
- **Depths probed (relative to model size N):** **early `N/4`**, **mid `N/2`**,
  **late `3N/4`**. Qwen3-8B (36 layers) -> `9 / 18 / 27`; Mistral-7B and
  Llama-3-8B-Instruct (32 layers) -> `8 / 16 / 24`.
- **7 configurations:** 3 single (`early`, `mid`, `late`), 3 double (`early+mid`,
  `early+late`, `mid+late`), 1 triple (`early+mid+late` = the deployed system).
- **Robustness:** every configuration is trained across **3 seeds**; we report
  **mean +/- std**. Metrics: **AUROC** (ranking quality) and **Accuracy / F1** at a
  fixed 0.5 threshold (the deployment operating point).
- **Cost:** the probe head is `Linear(input_width -> 256)`, so parameters scale with the
  number of concatenated layers (single ~ 1x, double ~ 2x, triple ~ 3x). All of it lives
  in the **tiny probe head**, not the frozen 8B backbone.

## 1. Per-model results (RAGTruth, 4000 rows, 3 seeds, mean +/- std)

Metrics shown as **AUROC / Accuracy / F1**; the per-metric winner in each model is in
**bold**. Probe-parameter counts are the combined CEV+IAV head sizes.

### Qwen3-8B (36 layers; early/mid/late = L9 / L18 / L27)
| Config | AUROC | Accuracy | F1 | Probe params |
|--------|:-----:|:--------:|:--:|:------------:|
| early `L9` | .8103 +/- .0131 | .7329 | .6915 | 4,262,660 |
| mid `L18` | .8466 | .7650 | **.7380** -> see mid+late | 4,262,660 |
| late `L27` | .8417 | **.7671** | .7234 | 4,262,660 |
| early+mid `L9+L18` | .8417 | .7612 | .7246 | 8,456,964 |
| early+late `L9+L27` | .8364 | .7562 | .7226 | 8,456,964 |
| **mid+late `L18+L27`** | **.8515** | .7642 | **.7395** | 8,456,964 |
| triple `L9+L18+L27` | .8427 | .7621 | .7289 | 12,651,268 |

*Best: AUROC `mid+late` .8515 | Accuracy `late` .7671 | F1 `mid+late` .7395. The triple
wins nothing; the best single layer is `mid` (.8466 AUROC) at ~1/2 the double's params.*

### Mistral-7B (32 layers; early/mid/late = L8 / L16 / L24)
| Config | AUROC | Accuracy | F1 | Probe params |
|--------|:-----:|:--------:|:--:|:------------:|
| early `L8` | .8113 +/- .0056 | .7292 | .6943 | 4,786,948 |
| **mid `L16`** | **.8306** | .7492 | .7159 | 4,786,948 |
| late `L24` | .8156 | .7313 | .6928 | 4,786,948 |
| early+mid `L8+L16` | .8274 | **.7546** | **.7182** | 9,505,540 |
| early+late `L8+L24` | .8214 | .7462 | .7082 | 9,505,540 |
| mid+late `L16+L24` | .8261 | .7500 | .7097 | 9,505,540 |
| triple `L8+L16+L24` | .8278 | .7442 | .7048 | 14,224,132 |

*Best: AUROC `mid` .8306 | Accuracy/F1 `early+mid` .7546 / .7182. The triple wins
nothing; the single `mid` layer takes AUROC at ~1/3 the triple's params.*

### Llama-3-8B-Instruct (32 layers; early/mid/late = L8 / L16 / L24)
| Config | AUROC | Accuracy | F1 | Probe params |
|--------|:-----:|:--------:|:--:|:------------:|
| early `L8` | .8086 | .7275 | .6868 | 4,786,948 |
| **mid `L16`** | **.8371 +/- .0112** | **.7596 +/- .0071** | **.7286 +/- .0074** | 4,786,948 |
| late `L24` | .8209 | .7417 | .7065 | 4,786,948 |
| early+mid `L8+L16` | .8284 | .7429 | .7027 | 9,505,540 |
| early+late `L8+L24` | .8268 | .7504 | .7129 | 9,505,540 |
| mid+late `L16+L24` | .8356 | .7542 | .7172 | 9,505,540 |
| triple `L8+L16+L24` | .8305 | .7417 | .7049 | 14,224,132 |

*Best: `mid` sweeps **all three** metrics (.8371 / .7596 / .7286) with the lowest
variance, at ~1/3 the triple's params. The triple wins nothing.*

## 2. Honest read

- **The single mid layer (`N/2`) is the strongest single depth in every model**, and is
  the best *overall* configuration on AUROC for Mistral and Llama, and on every metric
  for Llama (with the lowest seed-to-seed variance).
- **The 3-layer triple never wins a single metric in any model.** Its mean AUROC sits at
  or below the single mid layer everywhere, while it costs **~3x** the probe parameters.
- **Where a 2-layer probe wins, it always contains `mid`** (`mid+late` for Qwen AUROC/F1,
  `early+mid` for Mistral/Qwen accuracy-F1). The benefit comes from *including* the mid
  layer, not from stacking three.
- **Early (`N/4`) alone is consistently the weakest** depth, consistent with the
  hallucination signal accumulating with depth.
- **Differences are small relative to the seed std**, so we avoid over-claiming: the
  honest summary is that **mid is best-or-tied-best while being the cheapest**, and the
  triple's extra width buys nothing measurable.

## 3. System-agnostic best choice -> single MID layer (N/2)

To remove per-model noise we rank the 7 configurations **by mean AUROC within each model**
(1 = best; ties share the average rank) and average those ranks across the three models.
Lower is better.

| Config | Qwen rank | Mistral rank | Llama rank | **Avg rank** |
|--------|:---------:|:------------:|:----------:|:------------:|
| **mid** | 2 | 1 | 1 | **1.33** |
| mid+late | 1 | 4 | 2 | 2.33 |
| triple | 3 | 2 | 3 | 2.67 |
| early+mid | 4.5 | 3 | 4 | 3.83 |
| early+late | 6 | 5 | 5 | 5.33 |
| late | 4.5 | 6 | 6 | 5.50 |
| early | 7 | 7 | 7 | 7.00 |

**The single `mid` layer (N/2) is the best system-agnostic choice** (avg rank 1.33),
ahead of `mid+late` (2.33) and the `triple` (2.67). The triple is third on rank and most
expensive, so its ~3x parameters are **not justified**. (Section 5 recomputes this table
from the embedded data and asserts the winner is `mid`.)

In [7]:
# CPU-only: no torch / transformers / faiss / RAG / training here.
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless / CPU-only rendering
import matplotlib.pyplot as plt

OUTPUT_DIR = "compare_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Canonical configuration order and their depth class (cost tier).
CONFIGS = ["early", "mid", "late", "early+mid", "early+late", "mid+late", "triple"]
KIND = {
    "early": "single", "mid": "single", "late": "single",
    "early+mid": "double", "early+late": "double", "mid+late": "double",
    "triple": "triple",
}
MODEL_ORDER = ["Qwen3-8B", "Mistral-7B", "Llama-3-8B-Instruct"]

# Combined CEV+IAV probe-head parameters per cost tier, per model.
PARAMS = {
    "Qwen3-8B":            {"single": 4_262_660, "double": 8_456_964,  "triple": 12_651_268},
    "Mistral-7B":          {"single": 4_786_948, "double": 9_505_540,  "triple": 14_224_132},
    "Llama-3-8B-Instruct": {"single": 4_786_948, "double": 9_505_540,  "triple": 14_224_132},
}

# Executed 3-seed results: config -> (AUROC, Accuracy, F1).
RESULTS = {
    "Qwen3-8B": {
        "early":      (0.8103, 0.7329, 0.6915),
        "mid":        (0.8466, 0.7650, 0.7380),
        "late":       (0.8417, 0.7671, 0.7234),
        "early+mid":  (0.8417, 0.7612, 0.7246),
        "early+late": (0.8364, 0.7562, 0.7226),
        "mid+late":   (0.8515, 0.7642, 0.7395),
        "triple":     (0.8427, 0.7621, 0.7289),
    },
    "Mistral-7B": {
        "early":      (0.8113, 0.7292, 0.6943),
        "mid":        (0.8306, 0.7492, 0.7159),
        "late":       (0.8156, 0.7313, 0.6928),
        "early+mid":  (0.8274, 0.7546, 0.7182),
        "early+late": (0.8214, 0.7462, 0.7082),
        "mid+late":   (0.8261, 0.7500, 0.7097),
        "triple":     (0.8278, 0.7442, 0.7048),
    },
    "Llama-3-8B-Instruct": {
        "early":      (0.8086, 0.7275, 0.6868),
        "mid":        (0.8371, 0.7596, 0.7286),
        "late":       (0.8209, 0.7417, 0.7065),
        "early+mid":  (0.8284, 0.7429, 0.7027),
        "early+late": (0.8268, 0.7504, 0.7129),
        "mid+late":   (0.8356, 0.7542, 0.7172),
        "triple":     (0.8305, 0.7417, 0.7049),
    },
}

# Known AUROC seed-std (only the entries reported with +/- in the per-model notebooks);
# unknown entries are NaN so their error bars are simply not drawn.
AUROC_STD = {
    ("Qwen3-8B", "early"): 0.0131,
    ("Mistral-7B", "early"): 0.0056,
    ("Llama-3-8B-Instruct", "mid"): 0.0112,
}

rows = []
for model in MODEL_ORDER:
    for cfg in CONFIGS:
        auroc, acc, f1 = RESULTS[model][cfg]
        kind = KIND[cfg]
        rows.append({
            "model": model,
            "config": cfg,
            "kind": kind,
            "auroc": auroc,
            "auroc_std": AUROC_STD.get((model, cfg), np.nan),
            "accuracy": acc,
            "f1": f1,
            "params": PARAMS[model][kind],
        })

df = pd.DataFrame(rows)
df["config"] = pd.Categorical(df["config"], categories=CONFIGS, ordered=True)
df = df.sort_values(["model", "config"]).reset_index(drop=True)
print(df.to_string(index=False))

              model     config   kind  auroc  auroc_std  accuracy     f1   params
Llama-3-8B-Instruct      early single 0.8086        NaN    0.7275 0.6868  4786948
Llama-3-8B-Instruct        mid single 0.8371     0.0112    0.7596 0.7286  4786948
Llama-3-8B-Instruct       late single 0.8209        NaN    0.7417 0.7065  4786948
Llama-3-8B-Instruct  early+mid double 0.8284        NaN    0.7429 0.7027  9505540
Llama-3-8B-Instruct early+late double 0.8268        NaN    0.7504 0.7129  9505540
Llama-3-8B-Instruct   mid+late double 0.8356        NaN    0.7542 0.7172  9505540
Llama-3-8B-Instruct     triple triple 0.8305        NaN    0.7417 0.7049 14224132
         Mistral-7B      early single 0.8113     0.0056    0.7292 0.6943  4786948
         Mistral-7B        mid single 0.8306        NaN    0.7492 0.7159  4786948
         Mistral-7B       late single 0.8156        NaN    0.7313 0.6928  4786948
         Mistral-7B  early+mid double 0.8274        NaN    0.7546 0.7182  9505540
         Mistral

In [8]:
# Average-rank synthesis: rank configs by AUROC within each model (1 = best), then
# average across models. This is the quantitative basis for the recommendation.
auroc_pivot = df.pivot(index="config", columns="model", values="auroc").loc[CONFIGS, MODEL_ORDER]

# Rank 1 = highest AUROC; ties get the average rank (method="average").
rank_per_model = auroc_pivot.rank(axis=0, ascending=False, method="average")
avg_rank = rank_per_model.mean(axis=1).sort_values()

rank_table = rank_per_model.copy()
rank_table["avg_rank"] = rank_per_model.mean(axis=1)
rank_table = rank_table.sort_values("avg_rank")
print("Average AUROC rank across models (1 = best):")
print(rank_table.round(2).to_string())

best_config = avg_rank.index[0]
print(f"\nSystem-agnostic best configuration: {best_config!r} (avg rank {avg_rank.iloc[0]:.2f})")
assert best_config == "mid", f"expected mid to win, got {best_config!r}"
print("single MID layer (N/2) is the best system-agnostic choice.")

Average AUROC rank across models (1 = best):
model       Qwen3-8B  Mistral-7B  Llama-3-8B-Instruct  avg_rank
config                                                         
mid              2.0         1.0                  1.0      1.33
mid+late         1.0         4.0                  2.0      2.33
triple           3.0         2.0                  3.0      2.67
early+mid        4.5         3.0                  4.0      3.83
early+late       6.0         5.0                  5.0      5.33
late             4.5         6.0                  6.0      5.50
early            7.0         7.0                  7.0      7.00

System-agnostic best configuration: 'mid' (avg rank 1.33)
single MID layer (N/2) is the best system-agnostic choice.


In [9]:
# Figure 1: grouped AUROC bars by configuration x model, with seed error bars
# (whiskers only where a seed-std was reported).
x = np.arange(len(CONFIGS))
width = 0.26
colors = {"Qwen3-8B": "#4C72B0", "Mistral-7B": "#DD8452", "Llama-3-8B-Instruct": "#55A868"}

fig, ax = plt.subplots(figsize=(13, 6))
for i, model in enumerate(MODEL_ORDER):
    sub = df[df["model"] == model].set_index("config").loc[CONFIGS]
    yerr = np.nan_to_num(sub["auroc_std"].to_numpy(dtype=float), nan=0.0)
    ax.bar(x + (i - 1) * width, sub["auroc"], width, label=model, color=colors[model],
           yerr=yerr, capsize=3, error_kw={"elinewidth": 1.2})

ax.set_xticks(x)
ax.set_xticklabels(CONFIGS, rotation=20, ha="right")
ax.set_ylabel("Fused AUROC (mean over 3 seeds)")
ax.set_ylim(0.79, 0.86)
ax.set_title("Layer-wise probe AUROC by configuration and model")
ax.axvline(0.5, color="grey", lw=0.8, ls=":")  # visual hint between configs
ax.legend(title="Model", loc="upper left")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
p = os.path.join(OUTPUT_DIR, "layerwise_auroc_grouped_by_config.png")
fig.savefig(p, dpi=150, bbox_inches="tight")
plt.close(fig)
print("saved", p)

saved compare_output/layerwise_auroc_grouped_by_config.png


In [10]:
# Figure 2: (a) single-layer early/mid/late comparison; (b) best AUROC achievable at
# each cost tier (best single vs best double vs the triple).
singles = ["early", "mid", "late"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# (a) single-layer depth sweep
xs = np.arange(len(singles))
for i, model in enumerate(MODEL_ORDER):
    sub = df[(df["model"] == model) & (df["config"].isin(singles))].set_index("config").loc[singles]
    yerr = np.nan_to_num(sub["auroc_std"].to_numpy(dtype=float), nan=0.0)
    axes[0].bar(xs + (i - 1) * 0.26, sub["auroc"], 0.26, label=model, color=colors[model],
                yerr=yerr, capsize=3)
axes[0].set_xticks(xs)
axes[0].set_xticklabels(["early (N/4)", "mid (N/2)", "late (3N/4)"])
axes[0].set_ylabel("Fused AUROC")
axes[0].set_ylim(0.79, 0.855)
axes[0].set_title("(a) Single-layer depth sweep\nmid is the strongest single depth in every model")
axes[0].legend(fontsize=8)
axes[0].grid(axis="y", alpha=0.3)

# (b) best AUROC per cost tier
tiers = ["single", "double", "triple"]
xt = np.arange(len(tiers))
for i, model in enumerate(MODEL_ORDER):
    best = [df[(df["model"] == model) & (df["kind"] == t)]["auroc"].max() for t in tiers]
    axes[1].plot(xt, best, marker="o", label=model, color=colors[model])
    # annotate which config achieved the single-tier best (the recommendation anchor)
axes[1].set_xticks(xt)
axes[1].set_xticklabels(["best single", "best double", "triple"])
axes[1].set_ylabel("Best fused AUROC at this tier")
axes[1].set_title("(b) Best AUROC per cost tier\nadding layers past 'best single' barely moves AUROC")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.tight_layout()
p = os.path.join(OUTPUT_DIR, "layerwise_single_layer_and_best_per_depth.png")
fig.savefig(p, dpi=150, bbox_inches="tight")
plt.close(fig)
print("saved", p)

saved compare_output/layerwise_single_layer_and_best_per_depth.png


In [11]:
# Figure 3: (a) AUROC heatmap (model x config); (b) AUROC vs probe parameters scatter.
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# (a) heatmap
heat = auroc_pivot.T.loc[MODEL_ORDER, CONFIGS]  # rows=model, cols=config
im = axes[0].imshow(heat.to_numpy(), aspect="auto", cmap="viridis")
axes[0].set_xticks(np.arange(len(CONFIGS)))
axes[0].set_xticklabels(CONFIGS, rotation=20, ha="right")
axes[0].set_yticks(np.arange(len(MODEL_ORDER)))
axes[0].set_yticklabels(MODEL_ORDER)
for r in range(heat.shape[0]):
    for c in range(heat.shape[1]):
        axes[0].text(c, r, f"{heat.iat[r, c]:.3f}", ha="center", va="center",
                     color="white", fontsize=8)
axes[0].set_title("(a) Fused AUROC heatmap (model x config)")
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04, label="AUROC")

# (b) AUROC vs params scatter, marker per kind, color per model
markers = {"single": "o", "double": "s", "triple": "^"}
for model in MODEL_ORDER:
    sub = df[df["model"] == model]
    for kind in ["single", "double", "triple"]:
        s = sub[sub["kind"] == kind]
        axes[1].scatter(s["params"] / 1e6, s["auroc"], marker=markers[kind],
                        color=colors[model], s=70,
                        edgecolor="black", linewidth=0.4)
    # highlight this model's mid (the recommendation)
    midrow = sub[sub["config"] == "mid"]
    axes[1].scatter(midrow["params"] / 1e6, midrow["auroc"], marker="*",
                    color=colors[model], s=320, edgecolor="black", linewidth=0.6,
                    zorder=5)

# legend proxies
from matplotlib.lines import Line2D
kind_handles = [Line2D([0], [0], marker=markers[k], color="grey", linestyle="",
                       markersize=8, label=k) for k in ["single", "double", "triple"]]
model_handles = [Line2D([0], [0], marker="o", color=colors[m], linestyle="",
                        markersize=8, label=m) for m in MODEL_ORDER]
star_handle = [Line2D([0], [0], marker="*", color="grey", linestyle="", markersize=14,
                      label="mid (recommended)")]
axes[1].legend(handles=kind_handles + model_handles + star_handle, fontsize=8, loc="lower right")
axes[1].set_xlabel("Probe parameters (millions)")
axes[1].set_ylabel("Fused AUROC")
axes[1].set_title("(b) AUROC vs probe size\nthe ~3x-param triple buys no AUROC over the single mid layer")
axes[1].grid(alpha=0.3)

fig.tight_layout()
p = os.path.join(OUTPUT_DIR, "layerwise_auroc_heatmap_and_params_scatter.png")
fig.savefig(p, dpi=150, bbox_inches="tight")
plt.close(fig)
print("saved", p)

saved compare_output/layerwise_auroc_heatmap_and_params_scatter.png


## 4. Synthesis & recommendation

**Recommendation: deploy the single MID layer (`N/2`).** Across all three backbones the
single mid-depth probe is the best system-agnostic configuration (avg AUROC rank 1.33),
it wins Llama-3-8B-Instruct on every metric with the lowest variance, and it is the
**cheapest** option (~4.3-4.8M probe params). The 3-layer triple never won a single
metric in any model yet costs **~3x** the parameters, so it is **not justified**; where a
2-layer probe occasionally edges ahead on a fixed-threshold metric, it always contains the
mid layer, confirming the signal lives at `N/2`.

### Exact config switch (already applied to the three main pipeline notebooks)
In the `Config` dataclass:

```python
# before
probe_concat_n_layers: int = 3   # early/mid/late concat (deployed triple)
# after
probe_concat_n_layers: int = 1   # single MID layer (N/2)
```

The pipeline maps `probe_concat_n_layers <= 1` to `target_layers = [N // 2]`, i.e. the
mid layer (Qwen L18; Mistral / Llama L16). No other code change is needed.

> **Recalibrate `hallucination_threshold` after the switch.** Dropping from a 3-layer
> concat to a single layer changes the probe input width and the fused-score
> distribution, so the deployment decision threshold(s) must be **re-fit on the
> validation split** before use.

### Saved figures (in `compare_output/`)
- `layerwise_auroc_grouped_by_config.png` - grouped AUROC bars (config x model, error bars)
- `layerwise_single_layer_and_best_per_depth.png` - single-layer sweep + best-per-tier
- `layerwise_auroc_heatmap_and_params_scatter.png` - AUROC heatmap + AUROC-vs-params scatter